# Solución - Taller Integrador 1
## Evaluación y estrés de una cartera de crédito

**Curso:** Introducción a la programación para analítica financiera

Este notebook presenta una solución completa y comentada del taller. La idea no es únicamente obtener los resultados, sino mostrar una forma **ordenada, reusable y auditable** de resolver el problema con los contenidos trabajados hasta ahora: variables, condicionales, listas, ciclos, funciones, diccionarios y `pandas`.

### Archivos requeridos
Suba al entorno de Google Colab, en la misma carpeta del notebook:

- `solicitudes_credito.csv`
- `catalogo_productos.xlsx`

La base incluye deliberadamente problemas de calidad. La solución documenta cuáles pueden corregirse y cuáles impiden evaluar una solicitud.

## 1. Preparación del entorno

In [1]:
import os
import numpy as np
import pandas as pd

# Trabajaremos en la carpeta actual.
RUTA = "."
RUTA_RESULTADOS = "resultados"

# Creamos una carpeta para los archivos finales.
os.makedirs(RUTA_RESULTADOS, exist_ok=True)

ARCHIVO_SOLICITUDES = os.path.join(RUTA, "solicitudes_credito.csv")
ARCHIVO_CATALOGO = os.path.join(RUTA, "catalogo_productos.xlsx")

# Estas opciones solo facilitan la lectura de las tablas en pantalla.
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

## 2. Lectura y auditoría inicial

In [2]:
# Leemos los dos archivos entregados.
solicitudes_raw = pd.read_csv(ARCHIVO_SOLICITUDES)
catalogo_raw = pd.read_excel(ARCHIVO_CATALOGO)

# Revisamos dimensiones y primeras observaciones.
print("Solicitudes:", solicitudes_raw.shape)
print("Catálogo:", catalogo_raw.shape)

display(solicitudes_raw.head())
display(catalogo_raw)

# info() permite revisar tipos de datos y cantidad de no nulos.
solicitudes_raw.info()

# Conteo de faltantes por columna.
print("\nFaltantes en solicitudes:")
display(solicitudes_raw.isna().sum().to_frame("faltantes"))

# Revisamos duplicados del identificador.
print(
    "Duplicados por id_solicitud:",
    solicitudes_raw.duplicated(subset="id_solicitud").sum()
)

Solicitudes: (101, 12)
Catálogo: (5, 5)


,id_solicitud,fecha_solicitud,segmento,producto,ingreso_mensual,obligaciones_mensuales,monto_solicitado,tasa_ea,plazo_meses,score,antiguedad_meses,tiene_mora
0,SOL-0001,2026-08-25,Independiente,Libre Inversión,7389000.0,1693000.0,14240000,0.2207,18,806.0,7.0,No
1,SOL-0002,2026-08-11,Asalariado,Educativo,7275000.0,1002000.0,29630000,0.1205,36,815.0,7.0,No
2,SOL-0003,2026-08-02,Asalariado,Libre Inversión,9659000.0,2962000.0,54330000,0.1838,48,741.0,13.0,No
3,SOL-0004,2026-08-11,Asalariado,Vehículo,5034000.0,1785000.0,120580000,0.1926,24,673.0,21.0,No
4,SOL-0005,2026-08-08,asalariado,Libre Inversión,2275000.0,659000.0,57230000,0.2325,18,747.0,21.0,No


,producto,monto_maximo,tasa_referencia_ea,plazo_max_meses,comision_estudio_pct
0,Libre Inversión,50000000,0.205,60,0.012
1,Vehículo,120000000,16.800,84,0.009
2,Educativo,40000000,0.145,48,0.006
3,Microcrédito,25000000,0.278,36,0.015
4,Vehículo,120000000,16.800,84,0.009


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id_solicitud            101 non-null    object 
 1   fecha_solicitud         101 non-null    object 
 2   segmento                100 non-null    object 
 3   producto                101 non-null    object 
 4   ingreso_mensual         100 non-null    float64
 5   obligaciones_mensuales  100 non-null    float64
 6   monto_solicitado        101 non-null    int64  
 7   tasa_ea                 100 non-null    float64
 8   plazo_meses             101 non-null    int64  
 9   score                   100 non-null    float64
 10  antiguedad_meses        100 non-null    float64
 11  tiene_mora              100 non-null    object 
dtypes: float64(5), int64(2), object(5)
memory usage: 9.6+ KB

Faltantes en solicitudes:


,faltantes
id_solicitud,0
fecha_solicitud,0
segmento,1
producto,0
ingreso_mensual,1
obligaciones_mensuales,1
monto_solicitado,0
tasa_ea,1
plazo_meses,0
score,1


Duplicados por id_solicitud: 1


## 3. Funciones de limpieza

In [3]:
def normalizar_tasa(valor):
    """
    Lleva una tasa EA a escala decimal.

    0.205 -> 0.205
    20.5  -> 0.205
    """
    if pd.isna(valor):
        return np.nan

    valor = float(valor)

    if 0 < valor <= 1:
        return valor
    elif 1 < valor <= 100:
        return valor / 100
    else:
        return np.nan


def normalizar_mora(valor):
    """
    Convierte distintas formas de Sí/No a True/False.
    Si el contenido no se puede interpretar, devuelve NaN.
    """
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().lower()

    mapa_mora = {
        "sí": True,
        "si": True,
        "1": True,
        "true": True,
        "no": False,
        "0": False,
        "false": False
    }

    return mapa_mora.get(texto, np.nan)

## 4. Limpieza de solicitudes

In [4]:
# Copiamos la base para conservar el archivo original sin cambios.
solicitudes = solicitudes_raw.copy()

# ------------------------------------------------------------
# 4.1. Duplicados
# ------------------------------------------------------------

duplicados_id = solicitudes.duplicated(
    subset="id_solicitud",
    keep="first"
)

print("Filas duplicadas:", int(duplicados_id.sum()))

# Conservamos la primera aparición.
solicitudes = solicitudes.loc[~duplicados_id].copy()

# ------------------------------------------------------------
# 4.2. Variables de texto
# ------------------------------------------------------------

# Estandarizamos segmento.
solicitudes["segmento"] = (
    solicitudes["segmento"]
    .astype("string")
    .str.strip()
    .str.title()
)

# El segmento no entra en la regla de crédito.
# Por eso un faltante se conserva como categoría informativa.
solicitudes["segmento"] = solicitudes["segmento"].fillna(
    "Sin información"
)

# Estandarizamos producto.
solicitudes["producto"] = (
    solicitudes["producto"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Corregimos únicamente variantes ortográficas equivalentes.
solicitudes["producto"] = solicitudes["producto"].replace({
    "Microcredito": "Microcrédito",
    "Libre Inversion": "Libre Inversión"
})

# ------------------------------------------------------------
# 4.3. Variables numéricas
# ------------------------------------------------------------

columnas_numericas = [
    "ingreso_mensual",
    "obligaciones_mensuales",
    "monto_solicitado",
    "tasa_ea",
    "plazo_meses",
    "score",
    "antiguedad_meses"
]

for columna in columnas_numericas:
    solicitudes[columna] = pd.to_numeric(
        solicitudes[columna],
        errors="coerce"
    )

# Normalizamos la tasa y la mora.
solicitudes["tasa_ea"] = solicitudes["tasa_ea"].apply(
    normalizar_tasa
)

solicitudes["tiene_mora"] = solicitudes["tiene_mora"].apply(
    normalizar_mora
)

# Convertimos la fecha.
solicitudes["fecha_solicitud"] = pd.to_datetime(
    solicitudes["fecha_solicitud"],
    errors="coerce"
)

display(solicitudes.head())

Filas duplicadas: 1


,id_solicitud,fecha_solicitud,segmento,producto,ingreso_mensual,obligaciones_mensuales,monto_solicitado,tasa_ea,plazo_meses,score,antiguedad_meses,tiene_mora
0,SOL-0001,2026-08-25,Independiente,Libre Inversión,7389000.0,1693000.0,14240000,0.2207,18,806.0,7.0,False
1,SOL-0002,2026-08-11,Asalariado,Educativo,7275000.0,1002000.0,29630000,0.1205,36,815.0,7.0,False
2,SOL-0003,2026-08-02,Asalariado,Libre Inversión,9659000.0,2962000.0,54330000,0.1838,48,741.0,13.0,False
3,SOL-0004,2026-08-11,Asalariado,Vehículo,5034000.0,1785000.0,120580000,0.1926,24,673.0,21.0,False
4,SOL-0005,2026-08-08,Asalariado,Libre Inversión,2275000.0,659000.0,57230000,0.2325,18,747.0,21.0,False


## 5. Limpieza del catálogo y cruce

In [5]:
# Copiamos el catálogo.
catalogo = catalogo_raw.copy()

# Limpiamos la llave producto.
catalogo["producto"] = (
    catalogo["producto"]
    .astype("string")
    .str.strip()
    .str.title()
)

catalogo["producto"] = catalogo["producto"].replace({
    "Microcredito": "Microcrédito",
    "Libre Inversion": "Libre Inversión"
})

# Convertimos las variables numéricas.
columnas_catalogo = [
    "monto_maximo",
    "tasa_referencia_ea",
    "plazo_max_meses",
    "comision_estudio_pct"
]

for columna in columnas_catalogo:
    catalogo[columna] = pd.to_numeric(
        catalogo[columna],
        errors="coerce"
    )

# La tasa de referencia también puede venir en distinta escala.
catalogo["tasa_referencia_ea"] = (
    catalogo["tasa_referencia_ea"]
    .apply(normalizar_tasa)
)

# Revisamos productos duplicados después de limpiar espacios.
duplicados_catalogo = catalogo.duplicated(
    subset="producto",
    keep=False
)

display(
    catalogo.loc[
        duplicados_catalogo,
        ["producto", "monto_maximo", "tasa_referencia_ea"]
    ]
)

# El duplicado tiene las mismas condiciones, así que conservamos una fila.
catalogo = catalogo.drop_duplicates(
    subset="producto",
    keep="first"
).copy()

# Cruzamos solicitudes y catálogo.
trabajo = solicitudes.merge(
    catalogo,
    on="producto",
    how="left"
)

print("Filas antes del merge:", len(solicitudes))
print("Filas después del merge:", len(trabajo))

# Detectamos solicitudes cuyo producto no aparece en el catálogo.
sin_catalogo = trabajo["monto_maximo"].isna()

print("Solicitudes sin catálogo:", int(sin_catalogo.sum()))

display(
    trabajo.loc[
        sin_catalogo,
        ["id_solicitud", "producto"]
    ]
)

# Si falta la tasa de una solicitud, usamos la tasa de referencia
# del producto. La imputación queda explícita.
trabajo["tasa_ea"] = trabajo["tasa_ea"].fillna(
    trabajo["tasa_referencia_ea"]
)

,producto,monto_maximo,tasa_referencia_ea
1,Vehículo,120000000,0.168
4,Vehículo,120000000,0.168


Filas antes del merge: 100
Filas después del merge: 100
Solicitudes sin catálogo: 1


,id_solicitud,producto
94,SOL-0095,Crédito Verde


## 6. Registros utilizables

In [6]:
def identificar_problemas_calidad(fila):
    """
    Reúne los problemas que impiden calcular o aplicar la política.
    """
    problemas = []

    if pd.isna(fila["monto_maximo"]):
        problemas.append("Producto no catalogado")

    if pd.isna(fila["ingreso_mensual"]) or fila["ingreso_mensual"] <= 0:
        problemas.append("Ingreso faltante o no positivo")

    if (
        pd.isna(fila["obligaciones_mensuales"])
        or fila["obligaciones_mensuales"] < 0
    ):
        problemas.append("Obligaciones faltantes o negativas")

    if pd.isna(fila["monto_solicitado"]) or fila["monto_solicitado"] <= 0:
        problemas.append("Monto solicitado faltante o no positivo")

    if pd.isna(fila["tasa_ea"]) or not (0 < fila["tasa_ea"] <= 1):
        problemas.append("Tasa faltante o inválida")

    if pd.isna(fila["plazo_meses"]) or fila["plazo_meses"] <= 0:
        problemas.append("Plazo faltante o no positivo")

    if pd.isna(fila["score"]):
        problemas.append("Score faltante")

    if (
        pd.isna(fila["antiguedad_meses"])
        or fila["antiguedad_meses"] < 0
    ):
        problemas.append("Antigüedad faltante o negativa")

    if pd.isna(fila["tiene_mora"]):
        problemas.append("Estado de mora faltante/no reconocible")

    return " | ".join(problemas)


# Diagnosticamos fila por fila.
trabajo["motivo_calidad"] = trabajo.apply(
    identificar_problemas_calidad,
    axis=1
)

trabajo["registro_utilizable"] = trabajo["motivo_calidad"].eq("")

# Separamos los registros que no pueden evaluarse.
no_utilizables = trabajo.loc[
    ~trabajo["registro_utilizable"]
].copy()

cartera = trabajo.loc[
    trabajo["registro_utilizable"]
].copy()

print("Registros originales:", len(solicitudes_raw))
print("Duplicados retirados:", int(duplicados_id.sum()))
print("No utilizables:", len(no_utilizables))
print("Utilizables:", len(cartera))

display(
    no_utilizables[
        ["id_solicitud", "motivo_calidad"]
    ]
)

Registros originales: 101
Duplicados retirados: 1
No utilizables: 10
Utilizables: 90


,id_solicitud,motivo_calidad
13,SOL-0014,Antigüedad faltante o negativa
21,SOL-0022,Ingreso faltante o no positivo
46,SOL-0047,Plazo faltante o no positivo
54,SOL-0055,Ingreso faltante o no positivo
61,SOL-0062,Obligaciones faltantes o negativas
69,SOL-0070,Obligaciones faltantes o negativas
85,SOL-0086,Estado de mora faltante/no reconocible
91,SOL-0092,Monto solicitado faltante o no positivo
94,SOL-0095,Producto no catalogado
97,SOL-0098,Score faltante


## 7. Motor financiero reusable

In [7]:
def tasa_mensual(tasa_ea):
    """Convierte una tasa efectiva anual en tasa efectiva mensual."""
    return (1 + tasa_ea) ** (1 / 12) - 1


def cuota_fija(monto, tasa_ea, plazo_meses):
    """Calcula la cuota fija de un crédito."""
    i = tasa_mensual(tasa_ea)
    factor = 1 - (1 + i) ** (-plazo_meses)
    return monto * i / factor


def endeudamiento_proyectado(
    obligaciones_actuales,
    cuota_nueva,
    ingreso_mensual
):
    """Calcula la carga financiera incluyendo el nuevo crédito."""
    return (
        obligaciones_actuales + cuota_nueva
    ) / ingreso_mensual


def clasificar_credito(
    score,
    endeudamiento,
    antiguedad_meses,
    tiene_mora,
    monto_solicitado,
    monto_maximo,
    politica
):
    """
    Devuelve APROBADA, REVISIÓN o NEGADA según el diccionario politica.
    """

    cumple_aprobada = (
        score >= politica["score_aprobado"]
        and endeudamiento <= politica["endeudamiento_aprobado"]
        and antiguedad_meses >= politica["antiguedad_aprobado"]
        and tiene_mora == False
        and monto_solicitado <= monto_maximo
    )

    cumple_revision = (
        score >= politica["score_revision"]
        and endeudamiento <= politica["endeudamiento_revision"]
        and antiguedad_meses >= politica["antiguedad_revision"]
        and tiene_mora == False
        and monto_solicitado <= monto_maximo
    )

    if cumple_aprobada:
        return "APROBADA"
    elif cumple_revision:
        return "REVISIÓN"
    else:
        return "NEGADA"


# Política original del taller.
politica_original = {
    "score_aprobado": 700,
    "endeudamiento_aprobado": 0.40,
    "antiguedad_aprobado": 12,
    "score_revision": 650,
    "endeudamiento_revision": 0.50,
    "antiguedad_revision": 6
}

# Calculamos cuota y endeudamiento.
cartera["cuota_estimada"] = cartera.apply(
    lambda fila: cuota_fija(
        fila["monto_solicitado"],
        fila["tasa_ea"],
        int(fila["plazo_meses"])
    ),
    axis=1
)

cartera["endeudamiento_proyectado"] = cartera.apply(
    lambda fila: endeudamiento_proyectado(
        fila["obligaciones_mensuales"],
        fila["cuota_estimada"],
        fila["ingreso_mensual"]
    ),
    axis=1
)

# Aplicamos la política.
cartera["decision_base"] = cartera.apply(
    lambda fila: clasificar_credito(
        fila["score"],
        fila["endeudamiento_proyectado"],
        fila["antiguedad_meses"],
        fila["tiene_mora"],
        fila["monto_solicitado"],
        fila["monto_maximo"],
        politica_original
    ),
    axis=1
)

display(
    cartera[
        [
            "id_solicitud",
            "monto_solicitado",
            "cuota_estimada",
            "endeudamiento_proyectado",
            "decision_base"
        ]
    ].head()
)

,id_solicitud,monto_solicitado,cuota_estimada,endeudamiento_proyectado,decision_base
0,SOL-0001,14240000,9.229748e+05,0.354036,REVISIÓN
1,SOL-0002,29630000,9.761156e+05,0.271906,REVISIÓN
2,SOL-0003,54330000,1.567474e+06,0.468938,NEGADA
3,SOL-0004,120580000,6.004914e+06,1.547460,NEGADA
4,SOL-0005,57230000,3.736388e+06,1.932039,NEGADA


## 8. Análisis de la cartera

In [8]:
# Número y porcentaje por decisión.
resumen_decision = (
    cartera["decision_base"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="solicitudes")
)

resumen_decision["porcentaje"] = (
    resumen_decision["solicitudes"] / len(cartera)
)

# Creamos monto aprobado: 0 para los casos que no estén APROBADOS.
cartera["monto_aprobado_base"] = np.where(
    cartera["decision_base"].eq("APROBADA"),
    cartera["monto_solicitado"],
    0
)

monto_total_solicitado = cartera["monto_solicitado"].sum()
monto_total_aprobado = cartera["monto_aprobado_base"].sum()

print(f"Monto total solicitado: ${monto_total_solicitado:,.0f}")
print(f"Monto total aprobado:   ${monto_total_aprobado:,.0f}")

display(resumen_decision)

# Resumen por segmento.
resumen_segmento = (
    cartera.groupby("segmento", as_index=False)
    .agg(
        solicitudes=("id_solicitud", "count"),
        monto_solicitado=("monto_solicitado", "sum"),
        monto_aprobado=("monto_aprobado_base", "sum"),
        endeudamiento_promedio=("endeudamiento_proyectado", "mean")
    )
)

# Resumen por producto.
resumen_producto = (
    cartera.groupby("producto", as_index=False)
    .agg(
        solicitudes=("id_solicitud", "count"),
        monto_solicitado=("monto_solicitado", "sum"),
        monto_aprobado=("monto_aprobado_base", "sum"),
        endeudamiento_promedio=("endeudamiento_proyectado", "mean")
    )
)

display(resumen_segmento)
display(resumen_producto)

# Tabla de doble entrada: segmento + producto frente a decisión.
tabla_segmento_producto = cartera.pivot_table(
    index=["segmento", "producto"],
    columns="decision_base",
    values="id_solicitud",
    aggfunc="count",
    fill_value=0
)

display(tabla_segmento_producto)

# Tres aprobadas con mayor endeudamiento.
top_aprobadas_endeudamiento = (
    cartera.loc[cartera["decision_base"].eq("APROBADA")]
    .sort_values("endeudamiento_proyectado", ascending=False)
    .head(3)
)

# Tres negadas con mayor monto solicitado.
top_negadas_monto = (
    cartera.loc[cartera["decision_base"].eq("NEGADA")]
    .sort_values("monto_solicitado", ascending=False)
    .head(3)
)

display(
    top_aprobadas_endeudamiento[
        [
            "id_solicitud",
            "segmento",
            "producto",
            "monto_solicitado",
            "endeudamiento_proyectado"
        ]
    ]
)

display(
    top_negadas_monto[
        [
            "id_solicitud",
            "segmento",
            "producto",
            "monto_solicitado",
            "score",
            "endeudamiento_proyectado",
            "tiene_mora"
        ]
    ]
)

Monto total solicitado: $3,944,810,000
Monto total aprobado:   $237,940,000


,decision,solicitudes,porcentaje
0,NEGADA,63,0.700000
1,REVISIÓN,15,0.166667
2,APROBADA,12,0.133333


,segmento,solicitudes,monto_solicitado,monto_aprobado,endeudamiento_promedio
0,Asalariado,36,1490770000,119240000,0.546397
1,Independiente,24,1271950000,20600000,0.544913
2,Pensionado,16,574690000,58780000,0.576564
3,Pyme,13,548900000,39320000,0.337761
4,Sin información,1,58500000,0,0.287318


,producto,solicitudes,monto_solicitado,monto_aprobado,endeudamiento_promedio
0,Educativo,16,354850000,89990000,0.291425
1,Libre Inversión,32,1073030000,109340000,0.588902
2,Microcrédito,15,313200000,23220000,0.388044
3,Vehículo,27,2203730000,15390000,0.641597


decision_base                    APROBADA  NEGADA  REVISIÓN
segmento        producto                                   
Asalariado      Educativo               1       4         2
                Libre Inversión         4      11         2
                Microcrédito            0       4         1
                Vehículo                0       5         2
Independiente   Educativo               0       5         0
                Libre Inversión         1       5         3
                Microcrédito            0       1         0
                Vehículo                0       7         2
Pensionado      Educativo               1       0         0
                Libre Inversión         0       3         1
                Microcrédito            2       3         1
                Vehículo                1       4         0
Pyme            Educativo               2       1         0
                Libre Inversión         0       2         0
                Microcrédito            0       3         0
                Vehículo                0       4         1
Sin información Vehículo                0       1         0

,id_solicitud,segmento,producto,monto_solicitado,endeudamiento_proyectado
58,SOL-0059,Independiente,Libre Inversión,20600000,0.398682
43,SOL-0044,Asalariado,Libre Inversión,12030000,0.387563
53,SOL-0054,Asalariado,Educativo,30500000,0.386938


,id_solicitud,segmento,producto,monto_solicitado,score,endeudamiento_proyectado,tiene_mora
9,SOL-0010,Independiente,Vehículo,135290000,714.0,0.595171,False
8,SOL-0009,Asalariado,Vehículo,134890000,814.0,0.627141,False
89,SOL-0090,Independiente,Vehículo,133730000,656.0,0.702887,False


## 9. Escenario de estrés

In [9]:
# Supuestos del escenario adverso.
escenario_estres = {
    "factor_ingreso": 0.88,
    "aumento_tasa_ea": 0.03
}

# Aplicamos los dos shocks.
cartera["ingreso_estres"] = (
    cartera["ingreso_mensual"]
    * escenario_estres["factor_ingreso"]
)

cartera["tasa_ea_estres"] = (
    cartera["tasa_ea"]
    + escenario_estres["aumento_tasa_ea"]
)

# Reutilizamos la función de cuota.
cartera["cuota_estres"] = cartera.apply(
    lambda fila: cuota_fija(
        fila["monto_solicitado"],
        fila["tasa_ea_estres"],
        int(fila["plazo_meses"])
    ),
    axis=1
)

# Reutilizamos la función de endeudamiento.
cartera["endeudamiento_estres"] = cartera.apply(
    lambda fila: endeudamiento_proyectado(
        fila["obligaciones_mensuales"],
        fila["cuota_estres"],
        fila["ingreso_estres"]
    ),
    axis=1
)

# Reutilizamos el mismo motor de decisión.
cartera["decision_estres"] = cartera.apply(
    lambda fila: clasificar_credito(
        fila["score"],
        fila["endeudamiento_estres"],
        fila["antiguedad_meses"],
        fila["tiene_mora"],
        fila["monto_solicitado"],
        fila["monto_maximo"],
        politica_original
    ),
    axis=1
)

# Identificamos cambios de categoría.
cartera["cambio_estres"] = (
    cartera["decision_base"]
    != cartera["decision_estres"]
)

cartera["monto_aprobado_estres"] = np.where(
    cartera["decision_estres"].eq("APROBADA"),
    cartera["monto_solicitado"],
    0
)

numero_cambios = int(cartera["cambio_estres"].sum())
monto_aprobado_estres = cartera["monto_aprobado_estres"].sum()

perdida_monto_aprobado = (
    monto_total_aprobado
    - monto_aprobado_estres
)

print("Solicitudes que cambian:", numero_cambios)
print(f"Monto aprobado base:   ${monto_total_aprobado:,.0f}")
print(f"Monto aprobado estrés: ${monto_aprobado_estres:,.0f}")
print(f"Pérdida de monto aprobado: ${perdida_monto_aprobado:,.0f}")

Solicitudes que cambian: 10
Monto aprobado base:   $237,940,000
Monto aprobado estrés: $112,190,000
Pérdida de monto aprobado: $125,750,000


## 10. Deterioro por segmento

In [10]:
# Ordenamos las decisiones de peor a mejor.
orden_decision = {
    "NEGADA": 0,
    "REVISIÓN": 1,
    "APROBADA": 2
}

cartera["nivel_base"] = cartera["decision_base"].map(
    orden_decision
)

cartera["nivel_estres"] = cartera["decision_estres"].map(
    orden_decision
)

# Una solicitud deteriora si baja de nivel.
cartera["deterioro_estres"] = (
    cartera["nivel_estres"]
    < cartera["nivel_base"]
)

deterioro_segmento = (
    cartera.groupby("segmento", as_index=False)
    .agg(
        solicitudes=("id_solicitud", "count"),
        deterioros=("deterioro_estres", "sum")
    )
)

deterioro_segmento["proporcion_deterioro"] = (
    deterioro_segmento["deterioros"]
    / deterioro_segmento["solicitudes"]
)

deterioro_segmento = deterioro_segmento.sort_values(
    "proporcion_deterioro",
    ascending=False
)

display(deterioro_segmento)

segmento_mayor_deterioro = deterioro_segmento.iloc[0]

print(
    "Mayor deterioro:",
    segmento_mayor_deterioro["segmento"],
    f"({segmento_mayor_deterioro['proporcion_deterioro']:.1%})"
)

,segmento,solicitudes,deterioros,proporcion_deterioro
0,Asalariado,36,7,0.194444
1,Independiente,24,2,0.083333
2,Pensionado,16,1,0.062500
3,Pyme,13,0,0.000000
4,Sin información,1,0,0.000000


Mayor deterioro: Asalariado (19.4%)


## 11. Crédito individual y tabla de amortización

In [11]:
# Seleccionamos automáticamente el APROBADO de mayor monto.
credito_seleccionado = (
    cartera.loc[cartera["decision_base"].eq("APROBADA")]
    .sort_values("monto_solicitado", ascending=False)
    .iloc[0]
)

display(
    credito_seleccionado[
        [
            "id_solicitud",
            "segmento",
            "producto",
            "monto_solicitado",
            "tasa_ea",
            "plazo_meses"
        ]
    ]
)


def construir_amortizacion(monto, tasa_ea, plazo_meses):
    """
    Construye una tabla de amortización usando una lista,
    diccionarios y un ciclo for.
    """
    cuota = cuota_fija(monto, tasa_ea, plazo_meses)
    i = tasa_mensual(tasa_ea)
    saldo = float(monto)

    filas = []

    for periodo in range(1, plazo_meses + 1):

        interes = saldo * i
        abono_capital = cuota - interes
        cuota_real = cuota

        # Ajuste final por precisión numérica.
        if periodo == plazo_meses or abono_capital > saldo:
            abono_capital = saldo
            cuota_real = interes + abono_capital

        saldo = max(0, saldo - abono_capital)

        filas.append({
            "periodo": periodo,
            "cuota": cuota_real,
            "interes": interes,
            "abono_capital": abono_capital,
            "saldo": saldo
        })

    return pd.DataFrame(filas)


tabla_amortizacion = construir_amortizacion(
    float(credito_seleccionado["monto_solicitado"]),
    float(credito_seleccionado["tasa_ea"]),
    int(credito_seleccionado["plazo_meses"])
)

display(tabla_amortizacion.head())
display(tabla_amortizacion.tail())

# Total de intereses.
total_intereses = tabla_amortizacion["interes"].sum()

# Encontramos con while el primer período con saldo inferior al 50%.
mitad_monto = credito_seleccionado["monto_solicitado"] * 0.50
posicion = 0

while (
    posicion < len(tabla_amortizacion)
    and tabla_amortizacion.loc[posicion, "saldo"] >= mitad_monto
):
    posicion += 1

if posicion < len(tabla_amortizacion):
    periodo_mitad = int(
        tabla_amortizacion.loc[posicion, "periodo"]
    )
else:
    periodo_mitad = None

porcentaje_interes_primera = (
    tabla_amortizacion.iloc[0]["interes"]
    / tabla_amortizacion.iloc[0]["cuota"]
)

porcentaje_interes_ultima = (
    tabla_amortizacion.iloc[-1]["interes"]
    / tabla_amortizacion.iloc[-1]["cuota"]
)

print(f"Total de intereses: ${total_intereses:,.0f}")
print("Saldo < 50% por primera vez en el período:", periodo_mitad)
print(f"Interés / primera cuota: {porcentaje_interes_primera:.2%}")
print(f"Interés / última cuota:  {porcentaje_interes_ultima:.2%}")

id_solicitud               SOL-0072
segmento                 Asalariado
producto            Libre Inversión
monto_solicitado           42560000
tasa_ea                      0.2048
plazo_meses                      36
Name: 71, dtype: object

,periodo,cuota,interes,abono_capital,saldo
0,1,1.555280e+06,665948.581707,889331.778650,4.167067e+07
1,2,1.555280e+06,652032.951147,903247.409211,4.076742e+07
2,3,1.555280e+06,637899.578706,917380.781652,3.985004e+07
3,4,1.555280e+06,623545.057314,931735.303043,3.891830e+07
4,5,1.555280e+06,608965.926591,946314.433767,3.797199e+07


,periodo,cuota,interes,abono_capital,saldo
31,32,1.555280e+06,116169.927263,1.439110e+06,5.985175e+06
32,33,1.555280e+06,93651.751703,1.461629e+06,4.523546e+06
33,34,1.555280e+06,70781.227762,1.484499e+06,3.039047e+06
34,35,1.555280e+06,47552.842144,1.507728e+06,1.531319e+06
35,36,1.555280e+06,23960.995285,1.531319e+06,0.000000e+00


Total de intereses: $13,430,093
Saldo < 50% por primera vez en el período: 21
Interés / primera cuota: 42.82%
Interés / última cuota:  1.54%


## 12. Cambio de política: 40% a 45%

In [12]:
# Copiamos la política y modificamos solamente un parámetro.
politica_45 = politica_original.copy()
politica_45["endeudamiento_aprobado"] = 0.45

# El motor de decisión es exactamente el mismo.
cartera["decision_politica_45"] = cartera.apply(
    lambda fila: clasificar_credito(
        fila["score"],
        fila["endeudamiento_proyectado"],
        fila["antiguedad_meses"],
        fila["tiene_mora"],
        fila["monto_solicitado"],
        fila["monto_maximo"],
        politica_45
    ),
    axis=1
)

cartera["monto_aprobado_45"] = np.where(
    cartera["decision_politica_45"].eq("APROBADA"),
    cartera["monto_solicitado"],
    0
)

# Indicadores de ambas políticas.
n_aprobadas_original = int(
    cartera["decision_base"].eq("APROBADA").sum()
)

n_aprobadas_45 = int(
    cartera["decision_politica_45"].eq("APROBADA").sum()
)

monto_aprobado_original = cartera["monto_aprobado_base"].sum()
monto_aprobado_45 = cartera["monto_aprobado_45"].sum()

endeudamiento_aprobadas_original = (
    cartera.loc[
        cartera["decision_base"].eq("APROBADA"),
        "endeudamiento_proyectado"
    ].mean()
)

endeudamiento_aprobadas_45 = (
    cartera.loc[
        cartera["decision_politica_45"].eq("APROBADA"),
        "endeudamiento_proyectado"
    ].mean()
)

comparacion_politica = pd.DataFrame({
    "politica": ["Original - 40%", "Propuesta - 45%"],
    "numero_aprobaciones": [n_aprobadas_original, n_aprobadas_45],
    "monto_aprobado": [monto_aprobado_original, monto_aprobado_45],
    "endeudamiento_promedio_aprobadas": [
        endeudamiento_aprobadas_original,
        endeudamiento_aprobadas_45
    ]
})

display(comparacion_politica)

,politica,numero_aprobaciones,monto_aprobado,endeudamiento_promedio_aprobadas
0,Original - 40%,12,237940000,0.294020
1,Propuesta - 45%,16,346530000,0.325778


## 13. Hallazgos y conclusión para el comité

In [13]:
# Tomamos el segmento con mayor deterioro calculado previamente.
segmento_mayor_deterioro = deterioro_segmento.iloc[0]

hallazgo_1 = (
    f"La cartera evaluable contiene {len(cartera)} solicitudes; "
    f"{n_aprobadas_original} quedan aprobadas bajo la política original, "
    f"por un monto de ${monto_aprobado_original:,.0f}."
)

hallazgo_2 = (
    f"Bajo estrés, {numero_cambios} solicitudes cambian de categoría y "
    f"el monto aprobado disminuye en ${perdida_monto_aprobado:,.0f}."
)

hallazgo_3 = (
    f"El segmento con mayor proporción de deterioro es "
    f"{segmento_mayor_deterioro['segmento']}, con "
    f"{segmento_mayor_deterioro['proporcion_deterioro']:.1%}."
)

print("1.", hallazgo_1)
print("2.", hallazgo_2)
print("3.", hallazgo_3)

# Conclusión de menos de 120 palabras.
conclusion_comite = (
    f"La flexibilización del límite de endeudamiento de 40% a 45% "
    f"eleva las aprobaciones de {n_aprobadas_original} a {n_aprobadas_45} "
    f"y aumenta el monto aprobado de ${monto_aprobado_original:,.0f} "
    f"a ${monto_aprobado_45:,.0f}. Al mismo tiempo, el endeudamiento "
    f"promedio de los aprobados pasa de "
    f"{endeudamiento_aprobadas_original:.1%} a "
    f"{endeudamiento_aprobadas_45:.1%}. El escenario de estrés muestra "
    f"que {numero_cambios} solicitudes deterioran su clasificación y se "
    f"pierden ${perdida_monto_aprobado:,.0f} de colocación aprobada. "
    f"Por tanto, la flexibilización incrementa la colocación, pero debería "
    f"acompañarse de seguimiento a los nuevos aprobados y a los segmentos "
    f"más sensibles al estrés."
)

print("\nCONCLUSIÓN:")
print(conclusion_comite)

1. La cartera evaluable contiene 90 solicitudes; 12 quedan aprobadas bajo la política original, por un monto de $237,940,000.
2. Bajo estrés, 10 solicitudes cambian de categoría y el monto aprobado disminuye en $125,750,000.
3. El segmento con mayor proporción de deterioro es Asalariado, con 19.4%.

CONCLUSIÓN:
La flexibilización del límite de endeudamiento de 40% a 45% eleva las aprobaciones de 12 a 16 y aumenta el monto aprobado de $237,940,000 a $346,530,000. Al mismo tiempo, el endeudamiento promedio de los aprobados pasa de 29.4% a 32.6%. El escenario de estrés muestra que 10 solicitudes deterioran su clasificación y se pierden $125,750,000 de colocación aprobada. Por tanto, la flexibilización incrementa la colocación, pero debería acompañarse de seguimiento a los nuevos aprobados y a los segmentos más sensibles al estrés.


## 14. Exportación de resultados

In [14]:
# Archivo 1: base completa evaluada.
archivo_cartera = os.path.join(
    RUTA_RESULTADOS,
    "cartera_evaluada.xlsx"
)

cartera.to_excel(
    archivo_cartera,
    index=False
)

# Archivo 2: libro con los principales resúmenes.
archivo_resumen = os.path.join(
    RUTA_RESULTADOS,
    "resumen_cartera.xlsx"
)

with pd.ExcelWriter(archivo_resumen) as writer:

    no_utilizables.to_excel(
        writer,
        sheet_name="Auditoria_excluidos",
        index=False
    )

    resumen_decision.to_excel(
        writer,
        sheet_name="Decisiones",
        index=False
    )

    resumen_segmento.to_excel(
        writer,
        sheet_name="Por_segmento",
        index=False
    )

    resumen_producto.to_excel(
        writer,
        sheet_name="Por_producto",
        index=False
    )

    tabla_segmento_producto.to_excel(
        writer,
        sheet_name="Segmento_producto"
    )

    deterioro_segmento.to_excel(
        writer,
        sheet_name="Deterioro_estres",
        index=False
    )

    comparacion_politica.to_excel(
        writer,
        sheet_name="Comparacion_politica",
        index=False
    )

    tabla_amortizacion.to_excel(
        writer,
        sheet_name="Amortizacion",
        index=False
    )

print("Archivos generados:")
print(archivo_cartera)
print(archivo_resumen)

Archivos generados:
resultados/cartera_evaluada.xlsx
resultados/resumen_cartera.xlsx


## Lectura pedagógica de la solución

La lógica central del taller es que **los datos no se corrigen para obtener un resultado conveniente**. Primero se documenta su calidad; después se separan los casos que realmente pueden evaluarse. Los montos superiores al máximo del producto, por ejemplo, no se eliminan: son observaciones válidas que la política debe clasificar.

El segundo principio es la **reutilización**. La tasa mensual, la cuota, el endeudamiento y la decisión se construyen como funciones. Los parámetros de la política y del escenario se guardan en diccionarios. Por eso el escenario de estrés y el cambio del 40% al 45% no exigen volver a escribir el programa.